# 02 | Patch tiling and Random Forest baseline

The exported GeoTIFF is split into six tiles. Each tile is cut into 256 x 256 patches (kept only where at least 30% of pixels have a valid label) and saved to Drive so they survive a Colab disconnect. A pixel-wise Random Forest is then trained as the baseline.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q rasterio

In [ ]:
import os, glob
import numpy as np
import rasterio
from rasterio.windows import Window

TILE_DIR = "/content/drive/MyDrive/forest_loss_project"
PATCH_DIR = os.path.join(TILE_DIR, "patches")
os.makedirs(PATCH_DIR, exist_ok=True)

PATCH = 256
MIN_VALID = 0.3

tile_paths = sorted(glob.glob(os.path.join(TILE_DIR, "kinabatangan_stack_2025_v2*.tif")))
print(len(tile_paths), "tiles")

## Create patches

Bands 1 to 6 are features and band 7 is the label. Reading is windowed so a whole tile never has to fit in memory.

In [ ]:
n_saved = 0
for t_idx, tp in enumerate(tile_paths):
    with rasterio.open(tp) as src:
        profile = src.profile
        for r in range(0, src.height - PATCH + 1, PATCH):
            for c in range(0, src.width - PATCH + 1, PATCH):
                win = Window(c, r, PATCH, PATCH)
                data = src.read(window=win).astype("float32")
                label = data[6]
                valid_frac = np.mean(~np.isnan(label))
                if valid_frac < MIN_VALID:
                    continue
                np.save(os.path.join(PATCH_DIR, f"patch_t{t_idx}_r{r}_c{c}.npy"), data)
                n_saved += 1
    print("tile", t_idx, "done, total patches so far:", n_saved)

## Label balance check

In [ ]:
patch_files = sorted(glob.glob(os.path.join(PATCH_DIR, "*.npy")))
fracs = []
for f in patch_files:
    lab = np.load(f)[6]
    fracs.append(np.nanmean(lab))
fracs = np.array(fracs)
print("Patches:", len(patch_files))
print("Mean loss fraction:", round(float(np.nanmean(fracs)), 4))

import matplotlib.pyplot as plt
plt.hist(fracs, bins=30)
plt.xlabel("Fraction of loss pixels in patch")
plt.ylabel("Patches")
plt.show()

## Train/test split by patch

Splitting by patch (not by pixel) stops neighbouring pixels of the same patch leaking between train and test.

In [ ]:
from sklearn.model_selection import train_test_split

train_files, test_files = train_test_split(patch_files, test_size=0.25, random_state=42)
print(len(train_files), "train patches,", len(test_files), "test patches")

In [ ]:
PIX_PER_PATCH = 200
rng = np.random.default_rng(42)


def build_dataset(files, pix=PIX_PER_PATCH):
    X, y = [], []
    for f in files:
        d = np.load(f)
        feats = d[:6].reshape(6, -1).T
        lab = d[6].reshape(-1)
        valid = ~np.isnan(lab) & ~np.isnan(feats).any(axis=1)
        idx = np.where(valid)[0]
        if len(idx) == 0:
            continue
        take = rng.choice(idx, size=min(pix, len(idx)), replace=False)
        X.append(feats[take])
        y.append(lab[take])
    return np.vstack(X), np.concatenate(y)


X_train, y_train = build_dataset(train_files)
X_test, y_test = build_dataset(test_files)
print(X_train.shape, X_test.shape, "positive rate:", y_train.mean().round(3))

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, cohen_kappa_score

rf = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print(confusion_matrix(y_test, pred))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall:   ", round(recall_score(y_test, pred), 3))
print("Kappa:    ", round(cohen_kappa_score(y_test, pred), 3))